#### **OptaLive: Final Match Data Analysis & QA Audit Report**
This notebook executes the end-to-end OptaLive pipeline for Premier League match ingestion, pre-match readiness verification, chronological event processing, automated QA validation, and CSV audit report generation.

In [1]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.data_loader import OptaLiveLoader
from src.preprocessing import OptaLivePreprocessor
from src.validation import OptaLiveValidator

print("Environment loaded successfully. Modules imported.")

Environment loaded successfully. Modules imported.


#### **Load Match Metadata & Target Match**

In [2]:
loader = OptaLiveLoader(raw_data_dir="../data/raw")

# Load available matches
df_matches = loader.load_matches()
print(f"Total EPL matches available in dataset: {len(df_matches)}")

# Target Match: Chelsea vs Arsenal (Match ID: 3754217)
match_id = 3754217
print(f"Targeting Match ID: {match_id}")

Total EPL matches available in dataset: 380
Targeting Match ID: 3754217


#### **Run Ingestion & Preprocessing Pipeline**

In [3]:
lineups_df = loader.load_lineup(match_id)
events_df = loader.load_events(match_id)

# Preprocess data
preprocessor = OptaLivePreprocessor(raw_events_df=events_df, lineup_df=lineups_df)
key_events_df = preprocessor.extract_key_events()
readiness_df = preprocessor.validate_pre_match_readiness()

print(f"Extracted {len(key_events_df)} key match events.")
readiness_df

Extracted 14 key match events.


,match_id,team_id,team_name,total_squad,missing_jersey_count,readiness_status,notes
0,3754217,1,Arsenal,18,0,Ready,Team sheet verified successfully against Opta ...
1,3754217,33,Chelsea,18,0,Ready,Team sheet verified successfully against Opta ...


#### **Run Automated QA Validation Engine**

In [4]:
validator = OptaLiveValidator(key_events_df=key_events_df, lineup_df=lineups_df)
validated_events_df = validator.validate_events()
qa_report = validator.generate_data_quality_report()

print("========================================")
print("     MATCH DATA-QUALITY REPORT     ")
print("========================================")
for k, v in qa_report.items():
    print(f"{k.replace('_', ' ').title():<30} : {v}")
print("========================================")

     MATCH DATA-QUALITY REPORT     
Total Events                   : 14
Valid Events                   : 14
Warnings                       : 0
Errors                         : 0
Quality Score                  : 100.0


#### **Inspect Validation Audit Feed Sample**

In [5]:
display_cols = ['minute', 'event_type', 'team_name', 'player_name', 'validation_status', 'issue_description']
validated_events_df[display_cols].head(10)

,minute,event_type,team_name,player_name,validation_status,issue_description
0,0,Starting XI,Chelsea,NaN,Valid,No issues detected.
1,0,Starting XI,Arsenal,NaN,Valid,No issues detected.
2,30,Card,Arsenal,Santiago Cazorla González,Valid,No issues detected.
3,45,Substitution,Arsenal,Francis Joseph Coquelin,Valid,No issues detected.
4,48,Card,Chelsea,Branislav Ivanović,Valid,No issues detected.
5,52,Goal,Chelsea,Kurt Happy Zouma,Valid,No issues detected.
6,62,Card,Chelsea,Oscar dos Santos Emboaba Júnior,Valid,No issues detected.
7,68,Substitution,Chelsea,Oscar dos Santos Emboaba Júnior,Valid,No issues detected.
8,70,Card,Arsenal,Calum Chambers,Valid,No issues detected.
9,74,Substitution,Arsenal,Alexis Alejandro Sánchez Sánchez,Valid,No issues detected.


#### **Export Final Audit Report to CSV**

In [6]:
reports_dir = "../reports"
os.makedirs(reports_dir, exist_ok=True)

output_path = os.path.join(reports_dir, f"final_qa_report_{match_id}.csv")
validated_events_df.to_csv(output_path, index=False)

print(f"Successfully saved final QA audit report to: {output_path}")

Successfully saved final QA audit report to: ../reports\final_qa_report_3754217.csv
